# Amazon Search / LTR Project — Week 3 Prompt V2
## Error-Driven Prompt Refinement + Revalidation

This notebook uses the completed 50-query human validation set to improve the Week-3 LLM query-understanding prompt.

### Why Prompt V2?
Prompt V1 achieved:
- Core Product Accuracy ≈ 0.82
- Brand Accuracy ≈ 0.90
- Category Accuracy ≈ 0.70
- Attribute F1 ≈ 0.716
- Mean rewrite quality ≈ 1.02 / 2

The main failure modes were:
1. Overusing `product_search` instead of `exact_product_search`, `use_case_search`, or `other`
2. Hallucinating a product/category from review-like fragments
3. Confusing brand vs product line / shade / model
4. Numeric parsing errors (`33.8 oz`, `18.75 oz`, `5.3`, `765`, etc.)
5. Query rewrites dropping exact identifiers, quantities, codes, or sizes

Prompt V2 explicitly addresses those cases.

### Required uploads
- `amazon_week1_v4_processed_data.zip`
- `amazon_week2_baseline_ranking_results.zip`
- `week3_manual_validation_50_completed.csv`

### Output
- V2 structured query results
- V1 vs V2 manual metric comparison
- V1 vs V2 BM25 rewrite comparison
- error table
- final V2 scorecard
- optional 200-query formal run

In [ ]:
!pip -q install pandas numpy pyarrow rank-bm25 scikit-learn tqdm requests matplotlib

## 1. Configuration

In [ ]:
MODEL = "llama3:8b"
OLLAMA_BASE_URL = "http://127.0.0.1:11434"

N_VALIDATION = 50
RUN_200_AFTER_VALIDATION = False
RANDOM_SEED = 42

TARGET_ATTRIBUTE_F1 = 0.75
TARGET_INTENT_ACCURACY = 0.60

## 2. Colab-safe Ollama setup

This uses the manual Linux package path rather than the installer script that failed earlier.

In [ ]:
import os
import time
import shutil
import subprocess
import requests
import platform
from pathlib import Path

def ollama_alive():
    try:
        return requests.get(
            f"{OLLAMA_BASE_URL}/api/tags",
            timeout=2
        ).ok
    except Exception:
        return False

def run_cmd(cmd):
    print("+", cmd)
    r = subprocess.run(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError(
            f"Command failed ({r.returncode}): {cmd}"
        )

if shutil.which("ollama") is None:
    machine = platform.machine().lower()

    if machine in ("x86_64", "amd64"):
        arch = "amd64"
    elif machine in ("aarch64", "arm64"):
        arch = "arm64"
    else:
        raise RuntimeError(
            f"Unsupported architecture: {machine}"
        )

    run_cmd("apt-get -qq update")
    run_cmd("apt-get -qq install -y zstd curl")

    url = (
        f"https://ollama.com/download/"
        f"ollama-linux-{arch}.tar.zst"
    )

    run_cmd(
        f"curl -fL --retry 5 --retry-delay 2 "
        f"{url} -o /tmp/ollama.tar.zst"
    )

    run_cmd(
        "tar --use-compress-program=unzstd "
        "-xf /tmp/ollama.tar.zst -C /usr"
    )

if not ollama_alive():
    env = os.environ.copy()
    env["OLLAMA_HOST"] = "127.0.0.1:11434"

    log = open("/tmp/ollama_week3_v2.log", "w")

    OLLAMA_PROCESS = subprocess.Popen(
        ["ollama", "serve"],
        stdout=log,
        stderr=subprocess.STDOUT,
        env=env
    )

    for _ in range(90):
        if ollama_alive():
            break
        time.sleep(1)

assert ollama_alive(), "Ollama server did not start."

print("Pulling/checking:", MODEL)

subprocess.run(
    ["ollama", "pull", MODEL],
    check=True
)

print("Ollama ready.")

## 3. Upload Week 1 / Week 2 / completed manual validation

In [ ]:
from pathlib import Path
import zipfile

EXPECTED = [
    "amazon_week1_v4_processed_data.zip",
    "amazon_week2_baseline_ranking_results.zip",
    "week3_manual_validation_50_completed.csv"
]

def running_in_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False

missing = [
    f for f in EXPECTED
    if not Path(f).exists()
]

if missing and running_in_colab():
    from google.colab import files
    print("Upload the three required files.")
    files.upload()

missing = [
    f for f in EXPECTED
    if not Path(f).exists()
]

if missing:
    raise FileNotFoundError(
        "Missing: " + ", ".join(missing)
    )

WORKDIR = Path("amazon_week3_prompt_v2")
INPUT_DIR = WORKDIR / "inputs"
RESULTS_DIR = WORKDIR / "results"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for zip_name in EXPECTED[:2]:
    target = INPUT_DIR / Path(zip_name).stem
    target.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(target)

print("Files ready.")

## 4. Load project data + gold manual annotations

In [ ]:
import pandas as pd
import numpy as np
import json
import re
from IPython.display import display

W1_DIR = INPUT_DIR / "amazon_week1_v4_processed_data"
W2_DIR = INPUT_DIR / "amazon_week2_baseline_ranking_results"

qpr = pd.read_csv(
    W1_DIR / "query_product_relevance.csv"
)

products = pd.read_parquet(
    W1_DIR / "cleaned_products_10k.parquet"
)

fixed_test = pd.read_csv(
    W2_DIR / "fixed_test_queries.csv"
)

manual_gold = pd.read_csv(
    "week3_manual_validation_50_completed.csv"
)

for df in [qpr, fixed_test, manual_gold]:
    df["query_id"] = df["query_id"].astype(str)

qpr["product_id"] = qpr["product_id"].astype(str)
products["product_id"] = products["product_id"].astype(str)

query_table = (
    qpr[["query_id", "query"]]
    .drop_duplicates("query_id")
    .merge(
        fixed_test[["query_id"]],
        on="query_id",
        how="inner"
    )
)

order = {
    qid: i
    for i, qid in enumerate(
        fixed_test["query_id"].tolist()
    )
}

query_table["_order"] = (
    query_table["query_id"].map(order)
)

query_table = (
    query_table
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

validation_ids = manual_gold["query_id"].tolist()

validation_queries = (
    query_table[
        query_table["query_id"].isin(
            validation_ids
        )
    ]
    .copy()
)

validation_queries["_order"] = (
    validation_queries["query_id"].map(
        {
            qid: i
            for i, qid in enumerate(
                validation_ids
            )
        }
    )
)

validation_queries = (
    validation_queries
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

print("Validation queries:", len(validation_queries))
display(
    manual_gold[
        [
            "query_id",
            "query",
            "true_intent",
            "true_core_product",
            "true_brand",
            "true_category",
            "true_attributes_json",
            "rewrite_quality_0_to_2"
        ]
    ].head(10)
)

# 5. Prompt V2

### Intent decision rules

**exact_product_search**
- query contains a unique product name, model, product line, shade/code, SKU-like identifier, or a brand + highly specific product
- even if the query is messy, specificity wins

**brand_search**
- brand + broad category only, with no unique model/product line

**use_case_search**
- user states a need/problem/outcome but no concrete product is named

**category_browse**
- broad category exploration without a specific item

**product_search**
- a generic product type is named with attributes, but no unique model/product identifier

**other**
- review fragments, subjective comments, or text that does not reveal a product/category with confidence

### Critical V2 rules
- Never infer a product category from a review fragment
- Preserve every explicit alphanumeric code, quantity, decimal, size, shade, model, and unit
- Do not convert `33.8 oz` to `8 oz`
- Do not split `18.75 oz`
- Do not interpret shade/model codes such as `5.3`, `765`, `PMS061`, `Surf-350` as quantities
- A token that looks like a color may actually be a brand/product line/model; do not infer color unless context supports it
- Rewriting must not delete exact identifiers

In [ ]:
QUERY_SCHEMA = {
    "type": "object",
    "properties": {
        "query": {"type": "string"},
        "intent": {
            "type": "string",
            "enum": [
                "exact_product_search",
                "product_search",
                "category_browse",
                "brand_search",
                "use_case_search",
                "compatibility_search",
                "other"
            ]
        },
        "core_product": {
            "type": ["string", "null"]
        },
        "brand": {
            "type": ["string", "null"]
        },
        "category": {
            "type": ["string", "null"]
        },
        "attributes": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "value": {"type": "string"},
                    "explicit": {"type": "boolean"}
                },
                "required": [
                    "name",
                    "value",
                    "explicit"
                ],
                "additionalProperties": False
            }
        },
        "price_min": {
            "type": ["number", "null"]
        },
        "price_max": {
            "type": ["number", "null"]
        },
        "use_case": {
            "type": ["string", "null"]
        },
        "implicit_requirements": {
            "type": "array",
            "items": {"type": "string"}
        },
        "rewritten_query": {
            "type": "string"
        }
    },
    "required": [
        "query",
        "intent",
        "core_product",
        "brand",
        "category",
        "attributes",
        "price_min",
        "price_max",
        "use_case",
        "implicit_requirements",
        "rewritten_query"
    ],
    "additionalProperties": False
}

PROMPT_V2 = """
You are a conservative e-commerce search query understanding system.

Return exactly one JSON object following the provided schema.

FIRST: CLASSIFY INTENT USING THESE RULES

1. exact_product_search
Use this when the query contains a highly specific product identity:
- product name or product line
- model number
- shade/code
- SKU-like identifier
- brand + highly specific product
- exact size/variant tied to a specific named product

2. brand_search
Use this only when a brand is present with a broad product/category,
but there is no unique model/product line/shade/code.

3. use_case_search
Use this when the user expresses a goal, problem, or desired outcome
but does not name a concrete product.

4. category_browse
Use this for broad category exploration.

5. product_search
Use this when a generic product type is clearly named with optional
attributes, but there is no unique product identity.

6. compatibility_search
Use this when compatibility with another device/product is central.

7. other
Use this for:
- review fragments
- subjective comments
- incomplete text
- statements with no identifiable product/category
- ambiguous text where a product cannot be inferred confidently

CRITICAL CONSERVATISM RULES

- NEVER invent a product category from a review fragment.
- NEVER infer brand from a color-like or adjective-like token without evidence.
- NEVER infer color solely from a brand/product-line name.
- A capitalized token may be a product line, model, or shade, not necessarily a brand.
- If brand identity is uncertain, use null.
- If product type is not stated or confidently inferable, use null.
- If the query is ambiguous, prefer "other" over hallucinating a product.

NUMERIC / IDENTIFIER PRESERVATION RULES

Preserve every explicit:
- decimal
- quantity
- size
- unit
- shade code
- model code
- alphanumeric identifier

Examples of tokens that must stay exact:
33.8 oz
18.75 oz
5.3
765
PMS061
Surf-350
1090
20g
3g

Do NOT:
- split decimals
- convert codes into quantities
- reinterpret shade/model numbers as prices or sizes
- drop quantities from rewritten queries

ATTRIBUTE RULES

- Extract only attributes supported by the query.
- Explicitly stated attributes use explicit=true.
- High-confidence inferred requirements may use explicit=false.
- Subjective words like nice, great, wonderful, perfect are not stable attributes.
- If a term could be a product-line name rather than a literal attribute,
  do not convert it into an attribute without evidence.

QUERY REWRITE RULES

The rewritten query must:
- preserve all exact product identifiers
- preserve brand if explicit
- preserve product line
- preserve shade/model/code
- preserve quantity
- preserve size/unit
- preserve important compatibility terms
- remove only conversational filler
- NEVER change the product type
- NEVER add a speculative category
- NEVER replace a specific product with a generic one

Return JSON only.
""".strip()

## 6. V2 inference

In [ ]:
import requests
import time

def validate_output(obj, original_query):
    if not isinstance(obj, dict):
        raise ValueError("Output is not JSON object.")

    required = QUERY_SCHEMA["required"]
    missing = [
        x for x in required
        if x not in obj
    ]

    if missing:
        raise ValueError(
            f"Missing fields: {missing}"
        )

    obj["query"] = original_query

    if not isinstance(
        obj.get("attributes"),
        list
    ):
        obj["attributes"] = []

    if not isinstance(
        obj.get("implicit_requirements"),
        list
    ):
        obj["implicit_requirements"] = []

    return obj

def understand_v2(query, retries=3):
    messages = [
        {
            "role": "system",
            "content": PROMPT_V2
        },
        {
            "role": "user",
            "content": f"Query: {query}"
        }
    ]

    last_error = None

    for attempt in range(1, retries + 1):
        start = time.perf_counter()

        payload = {
            "model": MODEL,
            "messages": messages,
            "stream": False,
            "format": QUERY_SCHEMA,
            "options": {
                "temperature": 0,
                "seed": RANDOM_SEED,
                "num_ctx": 4096
            }
        }

        try:
            r = requests.post(
                f"{OLLAMA_BASE_URL}/api/chat",
                json=payload,
                timeout=180
            )

            if r.status_code >= 400:
                payload["format"] = "json"
                r = requests.post(
                    f"{OLLAMA_BASE_URL}/api/chat",
                    json=payload,
                    timeout=180
                )

            r.raise_for_status()

            raw = r.json()["message"]["content"]
            obj = validate_output(
                json.loads(raw),
                query
            )

            return {
                "status": "ok",
                "result": obj,
                "latency_sec":
                    time.perf_counter() - start,
                "error": None
            }

        except Exception as e:
            last_error = repr(e)
            time.sleep(min(attempt * 2, 6))

    return {
        "status": "error",
        "result": None,
        "latency_sec": np.nan,
        "error": last_error
    }

## 7. Run Prompt V2 on the same 50 gold queries

In [ ]:
from tqdm.auto import tqdm

rows = []

for r in tqdm(
    validation_queries.itertuples(index=False),
    total=len(validation_queries),
    desc="Prompt V2 validation"
):
    out = understand_v2(r.query)
    obj = out["result"] or {}

    rows.append({
        "query_id": r.query_id,
        "query": r.query,
        "v2_intent": obj.get("intent"),
        "v2_core_product":
            obj.get("core_product"),
        "v2_brand": obj.get("brand"),
        "v2_category": obj.get("category"),
        "v2_attributes_json":
            json.dumps(
                obj.get("attributes", []),
                ensure_ascii=False
            ),
        "v2_price_min":
            obj.get("price_min"),
        "v2_price_max":
            obj.get("price_max"),
        "v2_use_case":
            obj.get("use_case"),
        "v2_implicit_requirements_json":
            json.dumps(
                obj.get(
                    "implicit_requirements",
                    []
                ),
                ensure_ascii=False
            ),
        "v2_rewritten_query":
            obj.get("rewritten_query"),
        "v2_latency_sec":
            out["latency_sec"],
        "v2_status":
            out["status"],
        "v2_error":
            out["error"]
    })

v2 = pd.DataFrame(rows)

v2.to_csv(
    RESULTS_DIR
    / "week3_prompt_v2_outputs_50.csv",
    index=False
)

display(v2.head(10))

## 8. Manual metrics — V1 vs V2

The same gold labels are used for both versions.

In [ ]:
def norm_text(x):
    if pd.isna(x):
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(x).strip().lower()
    )

def safe_json(x, default):
    if pd.isna(x) or str(x).strip() == "":
        return default

    try:
        return json.loads(x)
    except Exception:
        return default

def attr_set_from_pred(x):
    items = safe_json(x, [])
    out = set()

    if isinstance(items, list):
        for item in items:
            if isinstance(item, dict):
                n = norm_text(
                    item.get("name")
                )
                v = norm_text(
                    item.get("value")
                )
                if n:
                    out.add((n, v))

    return out

def attr_set_from_true(x):
    obj = safe_json(x, {})
    out = set()

    if isinstance(obj, dict):
        for k, v in obj.items():
            out.add(
                (
                    norm_text(k),
                    norm_text(v)
                )
            )

    return out

merged = manual_gold.merge(
    v2,
    on=["query_id", "query"],
    how="left"
)

def exact_acc(df, pred, truth):
    return np.mean([
        norm_text(p) == norm_text(t)
        for p, t in zip(
            df[pred],
            df[truth]
        )
    ])

def attribute_metrics(
    df,
    pred_col,
    true_col="true_attributes_json"
):
    tp = fp = fn = 0

    for _, row in df.iterrows():
        pred = attr_set_from_pred(
            row[pred_col]
        )

        truth = attr_set_from_true(
            row[true_col]
        )

        tp += len(pred & truth)
        fp += len(pred - truth)
        fn += len(truth - pred)

    precision = (
        tp / (tp + fp)
        if tp + fp
        else 0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn
        else 0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall
        else 0
    )

    return precision, recall, f1

v1_attr = attribute_metrics(
    merged,
    "pred_attributes_json"
)

v2_attr = attribute_metrics(
    merged,
    "v2_attributes_json"
)

comparison = pd.DataFrame({
    "metric": [
        "intent_accuracy",
        "core_product_accuracy",
        "brand_accuracy",
        "category_accuracy",
        "attribute_precision_micro",
        "attribute_recall_micro",
        "attribute_f1_micro"
    ],
    "v1": [
        exact_acc(
            merged,
            "pred_intent",
            "true_intent"
        ),
        exact_acc(
            merged,
            "pred_core_product",
            "true_core_product"
        ),
        exact_acc(
            merged,
            "pred_brand",
            "true_brand"
        ),
        exact_acc(
            merged,
            "pred_category",
            "true_category"
        ),
        v1_attr[0],
        v1_attr[1],
        v1_attr[2]
    ],
    "v2": [
        exact_acc(
            merged,
            "v2_intent",
            "true_intent"
        ),
        exact_acc(
            merged,
            "v2_core_product",
            "true_core_product"
        ),
        exact_acc(
            merged,
            "v2_brand",
            "true_brand"
        ),
        exact_acc(
            merged,
            "v2_category",
            "true_category"
        ),
        v2_attr[0],
        v2_attr[1],
        v2_attr[2]
    ]
})

comparison["absolute_delta"] = (
    comparison["v2"]
    - comparison["v1"]
)

comparison.to_csv(
    RESULTS_DIR
    / "week3_prompt_v1_vs_v2_manual_metrics.csv",
    index=False
)

display(comparison)

## 9. Failure analysis

In [ ]:
error_table = merged[[
    "query_id",
    "query",
    "true_intent",
    "pred_intent",
    "v2_intent",
    "true_core_product",
    "pred_core_product",
    "v2_core_product",
    "true_brand",
    "pred_brand",
    "v2_brand",
    "true_category",
    "pred_category",
    "v2_category",
    "true_attributes_json",
    "pred_attributes_json",
    "v2_attributes_json",
    "pred_rewritten_query",
    "v2_rewritten_query",
    "rewrite_quality_0_to_2",
    "notes"
]].copy()

error_table["v1_intent_correct"] = [
    norm_text(p) == norm_text(t)
    for p, t in zip(
        error_table["pred_intent"],
        error_table["true_intent"]
    )
]

error_table["v2_intent_correct"] = [
    norm_text(p) == norm_text(t)
    for p, t in zip(
        error_table["v2_intent"],
        error_table["true_intent"]
    )
]

error_table["v1_core_correct"] = [
    norm_text(p) == norm_text(t)
    for p, t in zip(
        error_table["pred_core_product"],
        error_table["true_core_product"]
    )
]

error_table["v2_core_correct"] = [
    norm_text(p) == norm_text(t)
    for p, t in zip(
        error_table["v2_core_product"],
        error_table["true_core_product"]
    )
]

error_table.to_csv(
    RESULTS_DIR
    / "week3_prompt_v2_error_analysis.csv",
    index=False
)

print("V2 intent errors:")
display(
    error_table[
        ~error_table[
            "v2_intent_correct"
        ]
    ][[
        "query",
        "true_intent",
        "v2_intent",
        "notes"
    ]].head(25)
)

print("V2 core-product errors:")
display(
    error_table[
        ~error_table[
            "v2_core_correct"
        ]
    ][[
        "query",
        "true_core_product",
        "v2_core_product",
        "notes"
    ]].head(25)
)

# 10. BM25 Original vs V1 Rewrite vs V2 Rewrite

We now test whether Prompt V2 improves retrieval behavior, not only extraction accuracy.

In [ ]:
from rank_bm25 import BM25Okapi
from sklearn.metrics import ndcg_score

def tokenize(text):
    if pd.isna(text):
        return []
    return re.findall(
        r"[a-z0-9]+",
        str(text).lower()
    )

products["title"] = (
    products["title"]
    .fillna("")
    .astype(str)
)

products["description"] = (
    products["description"]
    .fillna("")
    .astype(str)
)

products["bm25_document"] = (
    products["title"].str.strip()
    + " "
    + products["description"].str.strip()
).str.strip()

corpus = [
    tokenize(x)
    for x in tqdm(
        products["bm25_document"],
        desc="Tokenizing catalog"
    )
]

bm25 = BM25Okapi(corpus)

product_ids = products["product_id"].tolist()
pid_to_idx = {
    pid: i
    for i, pid in enumerate(product_ids)
}

def reciprocal_rank(labels):
    for i, y in enumerate(labels, start=1):
        if y >= 1:
            return 1 / i
    return 0.0

def recall_at_k(
    retrieved,
    relevant,
    k
):
    relevant = set(relevant)

    if not relevant:
        return np.nan

    return len(
        set(retrieved[:k])
        & relevant
    ) / len(relevant)

def score_query(q):
    return np.asarray(
        bm25.get_scores(
            tokenize(q)
        )
    )

def evaluate_one(
    query_id,
    query_text,
    scores,
    version
):
    judged = qpr[
        qpr["query_id"] == str(query_id)
    ].copy()

    judged["idx"] = (
        judged["product_id"]
        .map(pid_to_idx)
    )

    judged = judged[
        judged["idx"].notna()
    ].copy()

    judged["idx"] = (
        judged["idx"].astype(int)
    )

    judged["score"] = scores[
        judged["idx"].values
    ]

    ranked = judged.sort_values(
        "score",
        ascending=False
    )

    labels = (
        ranked["relevance_label"]
        .astype(float)
        .tolist()
    )

    y_true = (
        judged["relevance_label"]
        .astype(float)
        .to_numpy()
        .reshape(1, -1)
    )

    y_score = (
        judged["score"]
        .astype(float)
        .to_numpy()
        .reshape(1, -1)
    )

    ndcg10 = float(
        ndcg_score(
            y_true,
            y_score,
            k=10
        )
    )

    mrr = reciprocal_rank(labels)

    top_idx = np.argpartition(
        scores,
        -50
    )[-50:]

    top_idx = top_idx[
        np.argsort(
            scores[top_idx]
        )[::-1]
    ]

    retrieved = [
        product_ids[i]
        for i in top_idx
    ]

    relevant = set(
        judged.loc[
            judged["relevance_label"] >= 2,
            "product_id"
        ].astype(str)
    )

    return {
        "query_id": str(query_id),
        "version": version,
        "ndcg@10": ndcg10,
        "mrr": mrr,
        "recall@10": recall_at_k(
            retrieved,
            relevant,
            10
        ),
        "recall@50": recall_at_k(
            retrieved,
            relevant,
            50
        )
    }

In [ ]:
eval_rows = []

for row in tqdm(
    merged.itertuples(index=False),
    total=len(merged),
    desc="BM25 V1 vs V2"
):
    versions = {
        "original": row.query,
        "v1_rewrite":
            row.pred_rewritten_query,
        "v2_rewrite":
            row.v2_rewritten_query
    }

    for name, qtext in versions.items():
        if pd.isna(qtext):
            continue

        eval_rows.append(
            evaluate_one(
                row.query_id,
                qtext,
                score_query(qtext),
                name
            )
        )

retrieval_eval = pd.DataFrame(
    eval_rows
)

retrieval_summary = (
    retrieval_eval
    .groupby("version")[
        [
            "ndcg@10",
            "mrr",
            "recall@10",
            "recall@50"
        ]
    ]
    .mean()
    .reset_index()
)

retrieval_eval.to_csv(
    RESULTS_DIR
    / "week3_prompt_v2_retrieval_per_query.csv",
    index=False
)

retrieval_summary.to_csv(
    RESULTS_DIR
    / "week3_prompt_v2_retrieval_summary.csv",
    index=False
)

display(retrieval_summary)

## 11. Decision gate

Only expand to 200 queries if Prompt V2 materially improves the gold 50-query set.

Suggested gate:
- Attribute F1 ≥ 0.75
- Intent Accuracy ≥ 0.60
- No major Recall@50 regression vs original query

In [ ]:
v2_intent = float(
    comparison.loc[
        comparison["metric"]
        == "intent_accuracy",
        "v2"
    ].iloc[0]
)

v2_f1 = float(
    comparison.loc[
        comparison["metric"]
        == "attribute_f1_micro",
        "v2"
    ].iloc[0]
)

original_r50 = float(
    retrieval_summary.loc[
        retrieval_summary["version"]
        == "original",
        "recall@50"
    ].iloc[0]
)

v2_r50 = float(
    retrieval_summary.loc[
        retrieval_summary["version"]
        == "v2_rewrite",
        "recall@50"
    ].iloc[0]
)

gate = (
    v2_f1 >= TARGET_ATTRIBUTE_F1
    and v2_intent >= TARGET_INTENT_ACCURACY
    and v2_r50 >= original_r50 * 0.98
)

print("V2 Intent Accuracy:", v2_intent)
print("V2 Attribute F1:", v2_f1)
print("Original Recall@50:", original_r50)
print("V2 Rewrite Recall@50:", v2_r50)
print()
print(
    "PROMPT V2 GATE:",
    "PASS — ready for 200-query run"
    if gate
    else "NOT YET — inspect remaining errors before 200-query run"
)

# 12. Optional formal 200-query run

Set:

```python
RUN_200_AFTER_VALIDATION = True
```

at the top only after Prompt V2 passes the validation gate.

In [ ]:
if RUN_200_AFTER_VALIDATION:
    formal_queries = query_table.head(200).copy()

    formal_rows = []

    for r in tqdm(
        formal_queries.itertuples(index=False),
        total=len(formal_queries),
        desc="Prompt V2 formal 200-query run"
    ):
        out = understand_v2(r.query)
        obj = out["result"] or {}

        formal_rows.append({
            "query_id": r.query_id,
            "query": r.query,
            "intent": obj.get("intent"),
            "core_product":
                obj.get("core_product"),
            "brand":
                obj.get("brand"),
            "category":
                obj.get("category"),
            "attributes_json":
                json.dumps(
                    obj.get(
                        "attributes",
                        []
                    ),
                    ensure_ascii=False
                ),
            "price_min":
                obj.get("price_min"),
            "price_max":
                obj.get("price_max"),
            "use_case":
                obj.get("use_case"),
            "implicit_requirements_json":
                json.dumps(
                    obj.get(
                        "implicit_requirements",
                        []
                    ),
                    ensure_ascii=False
                ),
            "rewritten_query":
                obj.get("rewritten_query"),
            "latency_sec":
                out["latency_sec"],
            "status":
                out["status"],
            "error":
                out["error"]
        })

    formal_200 = pd.DataFrame(
        formal_rows
    )

    formal_200.to_csv(
        RESULTS_DIR
        / "week3_prompt_v2_formal_200.csv",
        index=False
    )

    formal_200.to_parquet(
        RESULTS_DIR
        / "week3_prompt_v2_formal_200.parquet",
        index=False
    )

    print("Formal 200-query run complete.")
else:
    print(
        "Skipped formal 200-query run. "
        "Validate Prompt V2 first."
    )

# 13. Export Prompt V2 results

In [ ]:
from datetime import datetime
import shutil

v1_f1 = float(
    comparison.loc[
        comparison["metric"]
        == "attribute_f1_micro",
        "v1"
    ].iloc[0]
)

report = f"""
# Amazon Search / LTR Project
## Week 3 — Prompt V2 Validation

### Motivation
Prompt V2 was created from the 50-query manual error analysis.

### Manual Evaluation
- V1 Intent Accuracy: {float(comparison.loc[comparison['metric']=='intent_accuracy','v1'].iloc[0]):.4f}
- V2 Intent Accuracy: {v2_intent:.4f}
- V1 Attribute F1: {v1_f1:.4f}
- V2 Attribute F1: {v2_f1:.4f}
- Attribute F1 target: >= {TARGET_ATTRIBUTE_F1:.2f}

### Retrieval
- Original Recall@50: {original_r50:.4f}
- V2 Rewrite Recall@50: {v2_r50:.4f}

### Validation Gate
{'PASS' if gate else 'NOT YET PASSED'}

### Prompt V2 Changes
1. Stricter intent routing
2. Review fragments -> other unless product is explicit
3. Preserve decimal values, quantities, model/shade codes
4. Separate brand/product-line/model concepts
5. Prevent speculative categories
6. Preserve exact identifiers during rewrite

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}
""".strip()

(
    RESULTS_DIR
    / "Amazon_Week3_PromptV2_Report.md"
).write_text(
    report + "\n",
    encoding="utf-8"
)

(
    RESULTS_DIR
    / "prompt_v2.txt"
).write_text(
    PROMPT_V2 + "\n",
    encoding="utf-8"
)

zip_path = shutil.make_archive(
    "amazon_week3_prompt_v2_results",
    "zip",
    root_dir=RESULTS_DIR
)

print(report)
print()
print("Created:", zip_path)

if running_in_colab():
    from google.colab import files
    files.download(zip_path)